# Publication copy

Notebook outputs were cleared for distribution. Scientific settings are preserved; see `../docs/REPRODUCIBILITY.md` before interpreting numerical intervals. Run with the notebook working directory set to `plot/`.


# 导入包与写标签

In [ ]:
import numpy as np
import emcee 
import corner
import matplotlib.pyplot as plt
from IPython.display import display, Math
import getdist
from getdist import MCSamples, plots
plt.rcParams["mathtext.fontset"] = "cm"  # 方案 1
%config InlineBackend.figure_format = 'retina'  

In [ ]:
labelsmath = ['logM_{PBH}',  \
            'logf_{PBH}', 'log\\lambda_{E}^{6-8}','log\\lambda_{E}^{9}', 'log\\lambda_{E}^{10}', 'log\\lambda_{E}^{11}',\
                'log\\lambda_{E}^{12.5}', 'log\\lambda_{E}^{13.5}','\\alpha_{\\star}', '\\beta_{\\star}', '\\epsilon_{\\star}^s',  \
                    '\\epsilon_{\\star}^i', 'M_c^s', 'M_c^i', '\\sigma_{M_{UV}}','Q']
labels = [ '$logM_{PBH}$',  \
            '$logf_{PBH}$', '$log\\lambda_{E}^{6-8}$','$log\\lambda_{E}^{9}$', '$log\\lambda_{E}^{10}$', '$log\\lambda_{E}^{11}$',\
                '$log\\lambda_{E}^{12.5}$', '$log\\lambda_{E}^{13.5}$','$\\alpha_{\\star}$', '$\\beta_{\\star}$', '$\\epsilon_{\\star}^s$',  \
                    '$\\epsilon_{\\star}^i$', '$M_c^s$', '$M_c^i$', '$\\sigma_{M_{UV}}$','$Q$']


# 读取文件

In [ ]:
# Portable posterior snapshot, exported with the original notebook's selection.
from pathlib import Path
_snapshot_path = Path('../posterior/posterior_samples.npz')
_snapshot = np.load(_snapshot_path, allow_pickle=False)
discard, thin = int(_snapshot['discard']), int(_snapshot['thin'])
chains = _snapshot['chains']
log_prob = _snapshot['log_prob']
move_fraction = _snapshot['move_fraction']
bad_walkers = _snapshot['bad_walkers']
keep = _snapshot['keep']
flatchain = chains.reshape(-1, chains.shape[-1])
flat_log_prob = log_prob.reshape(-1)
nsamples, nwalkers, ndim = chains.shape
reader = None
_full_chain_path = Path('../mcmc/emcee_UVLF_P18_a16_merged_z20.h5')
if _full_chain_path.exists():
    reader = emcee.backends.HDFBackend(str(_full_chain_path), read_only=True)
print('Posterior snapshot:', chains.shape, 'excluded walkers:', bad_walkers)


In [ ]:
acceptance = _snapshot['accepted'] / int(_snapshot['iteration'])
print('min:', acceptance.min())
print('median:', np.median(acceptance))
print('max:', acceptance.max())


# 计算统计量

## 计算均值、方差

### 16%-84%

In [ ]:
par_median = np.zeros(ndim)
par_low = np.zeros(ndim)
par_upp = np.zeros(ndim)
par_per = np.zeros(ndim)
for i in range(0,ndim):
    mc = np.percentile(flatchain[:,i],[5,32,95])
    par_median[i] = mc[1]
    par_low[i] = mc[0]
    par_upp[i] = mc[2]
    q = np.diff(mc)
    par_per[i] =abs((q[0]+q[1])/mc[1]*50)
    #print(q)
    txt = "\\mathrm{{{4}}} = {0:.5g}_{{-{1:.2g}}}^{{{2:.2g}}}{3:.2g}\\%"
    txt = txt.format(mc[1], q[0], q[1],par_per[i], labelsmath[i])
    display(Math(txt))
if reader is not None:
    reader.get_autocorr_time(tol=0)
else:
    print('Full-chain autocorrelation needs the original HDF5 chain; not estimated from thinned samples.')

### GetDist KDE 最短连续 68% 可信区间

In [ ]:
# 与第一个三角图一致的 GetDist 设置（用于 log lambda_E 参数）
selected_columns = list(range(ndim))
flatchain_subset = np.asarray(flatchain[:, selected_columns], dtype=float)

# 内部参数名使用简单字符串；LaTeX 只用于显示
param_names = [f"p{i}" for i in range(ndim)]
param_labels = [str(labelsmath[i]).strip("$") for i in selected_columns]

# 与绘图单元使用相同的物理硬边界
bounds = [
    (0, 14),
    (-12, 0),
    *[(-15, 5)] * 6,
    (-3, 0),
    (0, 3),
    *[(-3, 3)] * 3,
    (7, 15),
    (0.001, 3),
    (1.5, 2.5),
]
hard_ranges = dict(zip(param_names, bounds))

analysis_settings = {
    "smooth_scale_1D": -2,
    "smooth_scale_2D": -2,
    "boundary_correction_order": 1,
    "mult_bias_correction_order": 0,
    "fine_bins": 1024,
    "fine_bins_2D": 512,
    "contours": [0.68, 0.95, 0.99],
}

# flat_log_prob 是 emcee 保存的 log posterior；loglikes 只供 LikeStats 使用，
# 不会重新加权边缘后验密度。
samples = MCSamples(
    samples=flatchain_subset,
    names=param_names,
    labels=param_labels,
    ranges=hard_ranges,
    loglikes=-flat_log_prob,
    label=r"\phi_{\rm UV}",
    settings=analysis_settings,
)

from scipy.integrate import cumulative_trapezoid


def shortest_kde_interval(density, probability=0.68, n_scan=20001):
    """从 GetDist 的一维 KDE 计算最短连续可信区间。"""
    x = np.asarray(density.x, dtype=float)
    pdf = np.asarray(density.P, dtype=float)

    # 边界修正或样条插值可能产生极小负值
    pdf = np.clip(pdf, 0.0, None)

    if not 0.0 < probability < 1.0:
        raise ValueError("probability 必须在 0 和 1 之间")

    cdf = cumulative_trapezoid(pdf, x, initial=0.0)
    if cdf[-1] <= 0.0:
        raise ValueError("KDE 积分非正")
    cdf /= cdf[-1]

    # 搜索所有 KDE 概率质量恰好为 probability 的连续区间
    lower_mass = np.linspace(0.0, 1.0 - probability, n_scan)
    lower_candidates = np.interp(lower_mass, cdf, x)
    upper_candidates = np.interp(lower_mass + probability, cdf, x)

    best_index = np.argmin(upper_candidates - lower_candidates)
    lower = lower_candidates[best_index]
    upper = upper_candidates[best_index]
    enclosed = np.interp(upper, x, cdf) - np.interp(lower, x, cdf)

    return lower, upper, enclosed


par_best = np.zeros(ndim)
par_low = np.zeros(ndim)
par_upp = np.zeros(ndim)
par_mass = np.zeros(ndim)

for i, name in enumerate(param_names):
    label = labelsmath[i]
    density = samples.get1DDensity(name)

    low68, upp68, enclosed68 = shortest_kde_interval(
        density,
        probability=0.68,
    )
    best = density.x[np.argmax(density.P)]  # 同一条 KDE 的一维边缘后验众数

    par_best[i] = best
    par_low[i] = low68
    par_upp[i] = upp68
    par_mass[i] = enclosed68

    q_low = best - low68
    q_upp = upp68 - best

    txt = (
        "\\mathrm{{{4}}} = {0:.5g}_{{-{1:.4g}}}^{{+{2:.4g}}}"
        "\\;(\\mathrm{{KDE~mass}}={3:.2f}\\%)"
    )
    txt = (
        rf"\mathrm{{{label}}} = {best:.5g}"
        rf"_{{-{q_low:.4g}}}^{{+{q_upp:.4g}}}"
        rf"\quad (\mathrm{{lower}}={low68:.5g},"
        rf"\;\mathrm{{upper}}={upp68:.5g})"
        rf"\quad (\mathrm{{KDE\ mass}}={100.0 * enclosed68:.2f}\%)"
    )
    display(Math(txt))


In [ ]:
like_stats = samples.getLikeStats()
print(like_stats)

for i in range(ndim):
    name = param_names[i]
    label = labelsmath[i]

    par_like = samples.paramNames.parWithName(name)

    print(
        f"{label} 95% n-D confidence extrema:",
        par_like.ND_limit_bot[1],
        par_like.ND_limit_top[1],
    )

## 计算自相关时间

In [ ]:
# file_paths = [f'emcee_UVLF{i}.h5' for i in range(1,8)]
# readers_t = [emcee.backends.HDFBackend(path, read_only=True) for path in file_paths]
# chains_t = [reader.get_chain(discard=0, thin=1) for reader in readers_t]
# n_t = np.zeros((len(chains_t),2))
# n_t[:,0] = [chain.shape[0] for chain in chains_t]
# n_t[:,1] = [np.mean(reader.get_autocorr_time(tol = 0)) for reader in readers_t]

# 画图

## 画自相关时间演化图

In [ ]:
# plt.figure()
# plt.plot(n_t[:,0],n_t[:,0] / 50.0, "--k")
# plt.plot(n_t[:,0], n_t[:,1], 'o-', label='tau_mean')
# # 添加标题和标签
# plt.title('Autocorr time')
# plt.xlabel("step number")
# plt.ylabel(r"mean $\hat{\tau}$")
# plt.legend([r'${step\,number}/{50}$',r'mean $\hat{\tau}$'])
# plt.show()

## 链运动轨迹图

In [ ]:
fig, axes = plt.subplots(ndim, sharex=True,figsize=(10, 1.5 * ndim))
for i in range(ndim):
    ax = axes[i]
    ax.plot(chains[:,:,i], alpha=0.4)#alpha是透明度
    ax.set_ylabel(labels[i])
axes[-1].set_xlabel("Step")

## 画 corner 图

In [ ]:
import numpy as np
from getdist import MCSamples, plots

# --------------------------------------------------
# 1. 选取参数
# --------------------------------------------------
selected_columns = [2, 3, 4, 5, 6, 7]

flatchain_subset = np.asarray(
    flatchain[:, selected_columns],
    dtype=float
)

# 参数内部名称最好使用简单字符串，不要直接使用 LaTeX
param_names = [f"p{i}" for i in range(len(selected_columns))]

# LaTeX 仅作为显示标签，并去掉外层的 $
param_labels = [
    str(labelsmath[i]).strip("$")
    for i in selected_columns
]

# --------------------------------------------------
# 2. KDE 设置
# --------------------------------------------------
bounds = [
    # (-3, 14),
    # (0,14),
    # (-12, 0),
    (-15, 2),
    (-15, 2),
    (-15, 2),
    (-15, 2),
    (-15, 2),
    (-15, 2),
]

hard_ranges = {
    name: bound
    for name, bound in zip(param_names, bounds)
}

analysis_settings = {
    "smooth_scale_1D": -7,
    "smooth_scale_2D": -7,

    # 真正存在硬边界时，可以恢复线性边界修正
    "boundary_correction_order": 1,
    "mult_bias_correction_order": 0,

    "fine_bins": 1024,
    "fine_bins_2D": 512,
    "contours": [0.68, 0.95],
}

samples = MCSamples(
    samples=flatchain_subset,
    names=param_names,
    labels=param_labels,
    ranges=hard_ranges,
    #label=r"Eddington Ratio",
    settings=analysis_settings,
)

# --------------------------------------------------
# 3. 绘图样式
# --------------------------------------------------
gd_settings = plots.GetDistPlotSettings()

gd_settings.axes_labelsize = 35
gd_settings.axes_fontsize = 30
gd_settings.legend_fontsize = 25
gd_settings.axis_marker_lw = 1.5
gd_settings.linewidth = 1.8
gd_settings.linewidth_contour = 1.8
gd_settings.num_plot_contours = 2

# 1D 曲线归一化到峰值为 1，方便和直方图叠加比较
gd_settings.norm_1d_density = False

g = plots.get_subplot_plotter(
    settings=gd_settings,
    subplot_size=2.7,
)

g.triangle_plot(
    [samples],
    params=param_names,

    # 正确的图例参数是 legend_labels
    #legend_labels=[r"$Eddington Ratio$"],

    filled=True,
    contour_colors=["#084594"],
    contour_lws=[1.8],
    contour_args=[{"alpha": 0.65}],

    # 强烈非高斯或多峰时，单个区间标题容易造成误解
    title_limit=None,
)
plt.savefig('Eddington_Ratio_triangle_plot.pdf', bbox_inches='tight')
# for i in range(len(param_names)):
#     ax = g.subplots[i, i]
#     values = flatchain_subset[:, i]

#     counts, edges = np.histogram(
#         values,
#         bins=25,
#         density=True,
#     )

#     if np.max(counts) > 0:
#         counts = counts / np.max(counts)

#     ax.stairs(
#         counts,
#         edges,
#         color="0.35",
#         linewidth=1.0,
#         alpha=0.65,
#         label="Raw histogram" if i == 0 else None,
#         zorder=10,
#     )
from scipy.integrate import cumulative_trapezoid

for name, label in zip(param_names, param_labels):
    density = samples.get1DDensity(name)
    x = np.asarray(density.x)
    pdf = np.clip(np.asarray(density.P), 0.0, None)

    # 对绘图 KDE 积分并归一化
    cdf = cumulative_trapezoid(pdf, x, initial=0.0)
    assert np.isfinite(cdf[-1]) and cdf[-1] > 0, "KDE 积分无效"
    cdf /= cdf[-1]

    q16, q50, q84 = np.interp([0.16, 0.50, 0.84], cdf, x)

    print(
        f"{label}: "
        f"q16={q16:.6g}, q50={q50:.6g}, q84={q84:.6g}"
    )
    display(Math(
        rf"{label} = {q50:.5g}"
        rf"_{{-{q50-q16:.4g}}}^{{+{q84-q50:.4g}}}"
    ))
from IPython.display import display, Math

def shortest_kde_interval(density, probability=0.68, n_scan=20001):
    """从平滑 KDE 计算最短连续可信区间。"""
    x = np.asarray(density.x, dtype=float)
    pdf = np.clip(np.asarray(density.P, dtype=float), 0.0, None)

    if not 0.0 < probability < 1.0:
        raise ValueError("probability 必须在 0 和 1 之间")

    cdf = cumulative_trapezoid(pdf, x, initial=0.0)
    assert np.isfinite(cdf[-1]) and cdf[-1] > 0, "KDE 积分无效"
    cdf /= cdf[-1]

    lower_mass = np.linspace(0.0, 1.0 - probability, n_scan)
    lower_candidates = np.interp(lower_mass, cdf, x)
    upper_candidates = np.interp(lower_mass + probability, cdf, x)

    best_index = np.argmin(upper_candidates - lower_candidates)
    lower = lower_candidates[best_index]
    upper = upper_candidates[best_index]
    enclosed = (
        np.interp(upper, x, cdf)
        - np.interp(lower, x, cdf)
    )

    return lower, upper, enclosed


print("\n平滑 KDE：最佳值与最短连续 68% 可信区间")

for name, label in zip(param_names, param_labels):
    density = samples.get1DDensity(name)

    low68, upp68, enclosed68 = shortest_kde_interval(
        density, probability=0.68
    )

    # 最佳值：该参数的平滑一维边缘后验众数
    best = density.x[np.argmax(density.P)]
    err_low = best - low68
    err_upp = upp68 - best

    print(
        f"{label}: best={best:.6g}, "
        f"lower68={low68:.6g}, upper68={upp68:.6g}, "
        f"KDE mass={100.0 * enclosed68:.2f}%"
    )

    display(Math(
        rf"{label}_{{\mathrm{{mode}}}} = {best:.5g}"
        rf"\quad \mathrm{{shortest}}\ 68\%\ \mathrm{{CI}}"
        rf"=[{low68:.5g},\,{upp68:.5g}]"
        rf"\quad (\mathrm{{KDE\ mass}}={100.0 * enclosed68:.2f}\%)"
    ))

    # 众数位于区间内时，再显示常用的非对称误差形式
    if low68 <= best <= upp68:
        display(Math(
            rf"{label} = {best:.5g}"
            rf"_{{-{err_low:.4g}}}^{{+{err_upp:.4g}}}"
        ))
# from scipy.integrate import trapezoid

# print("\n平滑 KDE：后验均值与最短连续 68% 可信区间")

# for name, label in zip(param_names, param_labels):
#     density = samples.get1DDensity(name)

#     low68, upp68, enclosed68 = shortest_kde_interval(
#         density, probability=0.68
#     )

#     # 后验均值：对归一化后的平滑一维边缘后验积分
#     x = np.asarray(density.x, dtype=float)
#     pdf = np.clip(np.asarray(density.P, dtype=float), 0.0, None)
#     norm = trapezoid(pdf, x)
#     assert np.isfinite(norm) and norm > 0, "KDE 积分无效"

#     mean = trapezoid(x * pdf, x) / norm
#     assert np.isfinite(mean), "后验均值无效"

#     err_low = mean - low68
#     err_upp = upp68 - mean

#     print(
#         f"{label}: mean={mean:.6g}, "
#         f"lower68={low68:.6g}, upper68={upp68:.6g}, "
#         f"KDE mass={100.0 * enclosed68:.2f}%"
#     )

#     display(Math(
#         rf"\langle {label} \rangle = {mean:.5g}"
#         rf"\quad \mathrm{{shortest}}\ 68\%\ \mathrm{{CI}}"
#         rf"=[{low68:.5g},\,{upp68:.5g}]"
#         rf"\quad (\mathrm{{KDE\ mass}}={100.0 * enclosed68:.2f}\%)"
#     ))

#     # 均值位于区间内时，显示非对称误差形式
#     if low68 <= mean <= upp68:
#         display(Math(
#             rf"\langle {label} \rangle = {mean:.5g}"
#             rf"_{{-{err_low:.4g}}}^{{+{err_upp:.4g}}}"
#         ))

In [ ]:
# --------------------------------------------------
# 1. 选取参数
# --------------------------------------------------
selected_columns = [0, 1]

flatchain_subset = np.asarray(
    flatchain[:, selected_columns],
    dtype=float
)

# 参数内部名称最好使用简单字符串，不要直接使用 LaTeX
param_names = [f"p{i}" for i in range(len(selected_columns))]

# LaTeX 仅作为显示标签，并去掉外层的 $
param_labels = [
    str(labelsmath[i]).strip("$")
    for i in selected_columns
]

# --------------------------------------------------
# 2. KDE 设置
# --------------------------------------------------
bounds = [
    (0, 14),
    (-12, 0),
    (-15, 5),
    (-15, 5),
]

hard_ranges = {
    name: bound
    for name, bound in zip(param_names, bounds)
}

analysis_settings = {
    "smooth_scale_1D": -2,
    "smooth_scale_2D": -2,

    # 真正存在硬边界时，可以恢复线性边界修正
    "boundary_correction_order": 1,
    "mult_bias_correction_order": 0,

    "fine_bins": 1024,
    "fine_bins_2D": 512,
    "contours": [0.68, 0.95],
}

samples = MCSamples(
    samples=flatchain_subset,
    names=param_names,
    labels=param_labels,
    ranges=hard_ranges,
    label=r"\phi_{\rm UV}",
    settings=analysis_settings,
)

# --------------------------------------------------
# 3. 绘图样式
# --------------------------------------------------
gd_settings = plots.GetDistPlotSettings()

gd_settings.axes_labelsize = 20
gd_settings.axes_fontsize = 18
gd_settings.legend_fontsize = 20
gd_settings.axis_marker_lw = 1.5
gd_settings.linewidth = 1.8
gd_settings.linewidth_contour = 1.8
gd_settings.num_plot_contours = 2

# 1D 曲线归一化到峰值为 1，方便和直方图叠加比较
gd_settings.norm_1d_density = False

g = plots.get_subplot_plotter(
    settings=gd_settings,
    subplot_size=2.7,
)

g.triangle_plot(
    [samples],
    params=param_names,

    # 正确的图例参数是 legend_labels
    legend_labels=[r"$\phi_{\rm UV}$"],

    filled=True,
    contour_colors=["#084594"],
    contour_lws=[1.8],
    contour_args=[{"alpha": 0.65}],

    # 强烈非高斯或多峰时，单个区间标题容易造成误解
    title_limit=None,
)
for i in range(len(param_names)):
    ax = g.subplots[i, i]
    values = flatchain_subset[:, i]

    counts, edges = np.histogram(
        values,
        bins=25,
        density=True,
    )

    if np.max(counts) > 0:
        counts = counts / np.max(counts)

    ax.stairs(
        counts,
        edges,
        color="0.35",
        linewidth=1.0,
        alpha=0.65,
        label="Raw histogram" if i == 0 else None,
        zorder=10,
    )
from scipy.integrate import cumulative_trapezoid

for name, label in zip(param_names, param_labels):
    density = samples.get1DDensity(name)
    x = np.asarray(density.x)
    pdf = np.clip(np.asarray(density.P), 0.0, None)

    # 对绘图 KDE 积分并归一化
    cdf = cumulative_trapezoid(pdf, x, initial=0.0)
    assert np.isfinite(cdf[-1]) and cdf[-1] > 0, "KDE 积分无效"
    cdf /= cdf[-1]

    q16, q50, q84 = np.interp([0.16, 0.50, 0.95], cdf, x)

    print(
        f"{label}: "
        f"q16={q16:.6g}, q50={q50:.6g}, q84={q84:.6g}"
    )
    display(Math(
        rf"{label} = {q50:.5g}"
        rf"_{{-{q50-q16:.4g}}}^{{+{q84-q50:.4g}}}"
    ))
from IPython.display import display, Math

def shortest_kde_interval(density, probability=0.68, n_scan=20001):
    """从平滑 KDE 计算最短连续可信区间。"""
    x = np.asarray(density.x, dtype=float)
    pdf = np.clip(np.asarray(density.P, dtype=float), 0.0, None)

    if not 0.0 < probability < 1.0:
        raise ValueError("probability 必须在 0 和 1 之间")

    cdf = cumulative_trapezoid(pdf, x, initial=0.0)
    assert np.isfinite(cdf[-1]) and cdf[-1] > 0, "KDE 积分无效"
    cdf /= cdf[-1]

    lower_mass = np.linspace(0.0, 1.0 - probability, n_scan)
    lower_candidates = np.interp(lower_mass, cdf, x)
    upper_candidates = np.interp(lower_mass + probability, cdf, x)

    best_index = np.argmin(upper_candidates - lower_candidates)
    lower = lower_candidates[best_index]
    upper = upper_candidates[best_index]
    enclosed = (
        np.interp(upper, x, cdf)
        - np.interp(lower, x, cdf)
    )

    return lower, upper, enclosed


print("\n平滑 KDE：最佳值与最短连续 68% 可信区间")

for name, label in zip(param_names, param_labels):
    density = samples.get1DDensity(name)

    low68, upp68, enclosed68 = shortest_kde_interval(
        density, probability=0.68
    )

    # 最佳值：该参数的平滑一维边缘后验众数
    best = density.x[np.argmax(density.P)]
    err_low = best - low68
    err_upp = upp68 - best

    print(
        f"{label}: best={best:.6g}, "
        f"lower68={low68:.6g}, upper68={upp68:.6g}, "
        f"KDE mass={100.0 * enclosed68:.2f}%"
    )

    display(Math(
        rf"{label}_{{\mathrm{{mode}}}} = {best:.5g}"
        rf"\quad \mathrm{{shortest}}\ 68\%\ \mathrm{{CI}}"
        rf"=[{low68:.5g},\,{upp68:.5g}]"
        rf"\quad (\mathrm{{KDE\ mass}}={100.0 * enclosed68:.2f}\%)"
    ))

    # 众数位于区间内时，再显示常用的非对称误差形式
    if low68 <= best <= upp68:
        display(Math(
            rf"{label} = {best:.5g}"
            rf"_{{-{err_low:.4g}}}^{{+{err_upp:.4g}}}"
        ))

In [ ]:
selected_columns = [8,9,10, 11, 12, 13, 14,15]


flatchain_subset = np.asarray(
    flatchain[:, selected_columns],
    dtype=float
)

# 参数内部名称最好使用简单字符串，不要直接使用 LaTeX
param_names = [f"p{i}" for i in range(len(selected_columns))]

# LaTeX 仅作为显示标签，并去掉外层的 $
param_labels = [
    str(labelsmath[i]).strip("$")
    for i in selected_columns
]

# --------------------------------------------------
# 2. KDE 设置
# --------------------------------------------------
bounds = [
    (-3, 0),
    (0, 3),
    (-3, 3),
    (-3, 3),
    (-3, 3),
    (7, 15),
    (0.001, 3),
    (1.5, 2.5),
]

hard_ranges = {
    name: bound
    for name, bound in zip(param_names, bounds)
}

analysis_settings = {
    "smooth_scale_1D": -3,
    "smooth_scale_2D": -3,

    # 真正存在硬边界时，可以恢复线性边界修正
    "boundary_correction_order": 1,
    "mult_bias_correction_order": 0,

    "fine_bins": 2048,
    "fine_bins_2D": 512,
    "contours": [0.68, 0.95],
}

samples = MCSamples(
    samples=flatchain_subset,
    names=param_names,
    labels=param_labels,
    ranges=hard_ranges,
    label=r"\phi_{\rm UV}",
    settings=analysis_settings,
)

# --------------------------------------------------
# 3. 绘图样式
# --------------------------------------------------
gd_settings = plots.GetDistPlotSettings()

gd_settings.axes_labelsize = 20
gd_settings.axes_fontsize = 18
gd_settings.legend_fontsize = 20
gd_settings.axis_marker_lw = 1.5
gd_settings.linewidth = 1.8
gd_settings.linewidth_contour = 1.8
gd_settings.num_plot_contours = 2

# 1D 曲线归一化到峰值为 1，方便和直方图叠加比较
gd_settings.norm_1d_density = False

g = plots.get_subplot_plotter(
    settings=gd_settings,
    subplot_size=2.7,
)

g.triangle_plot(
    [samples],
    params=param_names,

    # 正确的图例参数是 legend_labels
    legend_labels=[r"$\phi_{\rm UV}$"],

    filled=True,
    contour_colors=["#084594"],
    contour_lws=[1.8],
    contour_args=[{"alpha": 0.65}],

    # 强烈非高斯或多峰时，单个区间标题容易造成误解
    title_limit=None,
)
for i in range(len(param_names)):
    ax = g.subplots[i, i]
    values = flatchain_subset[:, i]

    counts, edges = np.histogram(
        values,
        bins="fd",
        density=True,
    )

    if np.max(counts) > 0:
        counts = counts / np.max(counts)

    ax.stairs(
        counts,
        edges,
        color="black",
        linewidth=1.0,
        alpha=0.65,
        label="Raw histogram" if i == 0 else None,
        zorder=10,
    )
from scipy.integrate import cumulative_trapezoid

for name, label in zip(param_names, param_labels):
    density = samples.get1DDensity(name)
    x = np.asarray(density.x)
    pdf = np.clip(np.asarray(density.P), 0.0, None)

    # 对绘图 KDE 积分并归一化
    cdf = cumulative_trapezoid(pdf, x, initial=0.0)
    assert np.isfinite(cdf[-1]) and cdf[-1] > 0, "KDE 积分无效"
    cdf /= cdf[-1]

    q16, q50, q84 = np.interp([0.16, 0.50, 0.84], cdf, x)

    print(
        f"{label}: "
        f"q16={q16:.6g}, q50={q50:.6g}, q84={q84:.6g}"
    )
    display(Math(
        rf"{label} = {q50:.5g}"
        rf"_{{-{q50-q16:.4g}}}^{{+{q84-q50:.4g}}}"
    ))
from scipy.integrate import cumulative_trapezoid

for name, label in zip(param_names, param_labels):
    density = samples.get1DDensity(name)
    x = np.asarray(density.x)
    pdf = np.clip(np.asarray(density.P), 0.0, None)

    # 对绘图 KDE 积分并归一化
    cdf = cumulative_trapezoid(pdf, x, initial=0.0)
    assert np.isfinite(cdf[-1]) and cdf[-1] > 0, "KDE 积分无效"
    cdf /= cdf[-1]

    q16, q50, q84 = np.interp([0.16, 0.50, 0.84], cdf, x)

    print(
        f"{label}: "
        f"q16={q16:.6g}, q50={q50:.6g}, q84={q84:.6g}"
    )
    display(Math(
        rf"{label} = {q50:.5g}"
        rf"_{{-{q50-q16:.4g}}}^{{+{q84-q50:.4g}}}"
    ))
from IPython.display import display, Math

def shortest_kde_interval(density, probability=0.68, n_scan=20001):
    """从平滑 KDE 计算最短连续可信区间。"""
    x = np.asarray(density.x, dtype=float)
    pdf = np.clip(np.asarray(density.P, dtype=float), 0.0, None)

    if not 0.0 < probability < 1.0:
        raise ValueError("probability 必须在 0 和 1 之间")

    cdf = cumulative_trapezoid(pdf, x, initial=0.0)
    assert np.isfinite(cdf[-1]) and cdf[-1] > 0, "KDE 积分无效"
    cdf /= cdf[-1]

    lower_mass = np.linspace(0.0, 1.0 - probability, n_scan)
    lower_candidates = np.interp(lower_mass, cdf, x)
    upper_candidates = np.interp(lower_mass + probability, cdf, x)

    best_index = np.argmin(upper_candidates - lower_candidates)
    lower = lower_candidates[best_index]
    upper = upper_candidates[best_index]
    enclosed = (
        np.interp(upper, x, cdf)
        - np.interp(lower, x, cdf)
    )

    return lower, upper, enclosed


print("\n平滑 KDE：最佳值与最短连续 68% 可信区间")

for name, label in zip(param_names, param_labels):
    density = samples.get1DDensity(name)

    low68, upp68, enclosed68 = shortest_kde_interval(
        density, probability=0.68
    )

    # 最佳值：该参数的平滑一维边缘后验众数
    best = density.x[np.argmax(density.P)]
    err_low = best - low68
    err_upp = upp68 - best

    print(
        f"{label}: best={best:.6g}, "
        f"lower68={low68:.6g}, upper68={upp68:.6g}, "
        f"KDE mass={100.0 * enclosed68:.2f}%"
    )

    display(Math(
        rf"{label}_{{\mathrm{{mode}}}} = {best:.5g}"
        rf"\quad \mathrm{{shortest}}\ 68\%\ \mathrm{{CI}}"
        rf"=[{low68:.5g},\,{upp68:.5g}]"
        rf"\quad (\mathrm{{KDE\ mass}}={100.0 * enclosed68:.2f}\%)"
    ))

    # 众数位于区间内时，再显示常用的非对称误差形式
    if low68 <= best <= upp68:
        display(Math(
            rf"{label} = {best:.5g}"
            rf"_{{-{err_low:.4g}}}^{{+{err_upp:.4g}}}"
        ))

In [ ]:
# 选择指定列（例如第0、2、3列）
selected_columns = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11,12,13,14,15]
# 筛选数据、标签和真实值
flatchain_subset = flatchain[:, selected_columns]
labels_subset = [labels[i] for i in selected_columns]
#truths_subset = [true_params[i] for i in selected_columns]

# 绘制指定参数的Corner图
corner.corner(
    flatchain_subset,
    labels=labels_subset,
    #truths=truths_subset,
    show_titles=True,
    title_fmt=".2g",
    quantiles=[0.16, 0.5, 0.84]
)
plt.show()

In [ ]:
# 选择指定列（例如第0、2、3列）
selected_columns = [0, 1, 2, 3, 4, 5, 6,7]
# 筛选数据、标签和真实值
flatchain_subset = flatchain[:, selected_columns]
labels_subset = [labels[i] for i in selected_columns]
#truths_subset = [true_params[i] for i in selected_columns]

# 绘制指定参数的Corner图
corner.corner(
    flatchain_subset,
    labels=labels_subset,
    #truths=truths_subset,
    show_titles=True,
    title_fmt=".2g",
    quantiles=[0.16, 0.5, 0.84]
)
plt.show()

In [ ]:
# 选择指定列（例如第0、2、3列）
selected_columns = [8, 9, 10, 11,12,13,14,15]
# 筛选数据、标签和真实值
flatchain_subset = flatchain[:, selected_columns]
labels_subset = [labels[i] for i in selected_columns]
#truths_subset = [true_params[i] for i in selected_columns]

# 绘制指定参数的Corner图
corner.corner(
    flatchain_subset,
    labels=labels_subset,
    #truths=truths_subset,
    show_titles=True,
    title_fmt=".2g",
    quantiles=[0.16, 0.5, 0.84]
)
plt.show()